# Gulfstream walkthrough — Faker HMM panel

Same Graph **1** / Graph **2** tour as the DuckDB notebooks, on **Faker-generated** panel data: ~10 years of daily observations, 10 made-up features, **4 outer regimes** (2 repeat), each regime an **HMM mixing two multivariate normals**. No extra feature engineering — the simulated series are the features.

| Part | Focus |
|------|--------|
| A–C | PCA / kPCA / DMD — Graph 1 + Graph 2 |
| D | t-SNE — Graph 1 + Graph 2 |
| E | UMAP — Graph 1 + Graph 2 (optional) |
| F | Search: Binseg / BottomUp / WBS / BOCPD |
| G | Tests: energy / MMD / Hotelling / CUSUM |
| H | ESS window |
| I | Classical hard-label detectors + Graph 2 |
| J | Classical models as soft dimred |
| K | TFT dimred (optional) |
| L | ICA / FPCA / dynamic factor |
| M | Product: uncertainty / Excel / events / streaming / panel |
| N | Graph 2 retrain scores |
| — | Comparison vs **true** outer breakpoints |

Run top-to-bottom. Scores print covering / ARI / F1 against ground truth.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import polars as pl
from plotnine import aes, facet_wrap, geom_line, ggplot, labs, theme, theme_bw

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError("Run from the gulfstream repo (or notebooks/).")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT_DIR = ROOT / "outputs" / "notebooks" / "faker_hmm"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)
print("OUT_DIR =", OUT_DIR)


## 1. Generate Faker HMM panel

Ten years of business days, 10 Faker-named features, 4 outer regimes (2 repeat later). Each outer regime is a **2-state HMM** mixing two MVNs.


In [ ]:
from gulfstream.data.synth import generate_faker_hmm_panel
from gulfstream import load_features, plot_regimes
from gulfstream.common import frames

SOURCE_YAML = ROOT / "config" / "sources" / "notebook_faker_hmm.yaml"

# Public loader (features only)
features_df = load_features(str(SOURCE_YAML), project_root=ROOT)
print("loaded shape:", features_df.shape)

# Same seed → ground-truth outer regimes for scoring
truth = generate_faker_hmm_panel(
    n_years=10,
    n_features=10,
    n_regimes=4,
    n_repeating=2,
    seed=42,
    include_labels=False,
)
TRUE_LABELS = truth.labels
TRUE_BKPTS = truth.bkpts
REGIME_SEQUENCE = truth.regime_sequence
assert features_df.height == len(TRUE_LABELS)
print("regime_sequence:", REGIME_SEQUENCE)
print("true_bkpts:", TRUE_BKPTS)
print("features:", frames.feature_columns(features_df))
features_df.head()


## 2. Explore series + true regime shading


In [ ]:
plot_cols = frames.feature_columns(features_df)[:4]

pdf = features_df.select(["date", *plot_cols]).to_pandas()
pdf_long = pdf.melt(id_vars="date", var_name="feature", value_name="value")
(
    ggplot(pdf_long, aes("date", "value", color="feature"))
    + geom_line(alpha=0.85)
    + facet_wrap("~feature", scales="free_y", ncol=2)
    + theme_bw()
    + labs(title="Faker HMM panel (first 4 features)", x="", y="")
    + theme(figure_size=(10, 6), legend_position="none")
)


In [ ]:
# True outer-regime intervals for reference
from gulfstream.detection import time_index as bkpt_time
from gulfstream.common.results import SegmentResults

truth_res = SegmentResults(
    bkpts=TRUE_BKPTS,
    hierarchy={b: 1 for b in TRUE_BKPTS},
)
fig_ground_truth_outer_regimes = plot_regimes(
    features_df,
    truth_res,
    variables=plot_cols[:2],
    title="Ground truth outer regimes",
    mode="display",
    emit=False,
)
fig_ground_truth_outer_regimes
print("Outer regime occupancy:")
pl.DataFrame({"regime": TRUE_LABELS}).group_by("regime").len().sort("regime")


## 3. Shared helpers (public API)

Graph 1 via `run_single_segmentation`; Graph 2 via `refine_regimes`. Each run prints covering / ARI / F1 against **true** outer breakpoints.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream import (
    plot_regimes,
    refine_regimes,
    run_single_segmentation,
    seed_regimes_from_results,
)
from gulfstream.common import frames, utils
from gulfstream.common.options import (
    ClassicalDetector,
    DetectionBackend,
    SearchMethod,
    StatTest,
)
from gulfstream.metrics.evaluation import (
    adjusted_rand_index,
    breakpoint_precision_recall_f1,
    covering_metric,
)


def load_core_params(img_dir: Path) -> dict:
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method == "dmd":
        out["algo"]["dmd_stride"] = [5]
        out["algo"]["dmd_rolling_window"] = [20]
    elif method == "ica":
        out["algo"]["rank"] = [3]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["random_state"] = [42]
        out["algo"]["ica_max_iter"] = [200]
    elif method == "fpca":
        out["algo"]["rank_selection_method"] = ["explained_variance"]
        out["algo"]["threshold"] = [0.9]
        out["algo"]["fpca_smooth_window"] = [3]
    elif method == "nelson_siegel":
        out["algo"]["ns_lambda"] = [0.0609]
    elif method == "dynamic_factor":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["factor_order"] = [1]
        out["algo"]["df_maxiter"] = [30]
    elif method == "tsne":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["tsne_perplexity"] = [30.0]
        out["algo"]["tsne_n_iter"] = [250]
        out["algo"]["random_state"] = [42]
    elif method == "umap":
        out["algo"]["rank"] = [2]
        out["algo"]["rank_selection_method"] = ["user_specified"]
        out["algo"]["umap_num_neighbors"] = [15]
        out["algo"]["umap_min_dist"] = [0.1]
        out["algo"]["umap_metric"] = ["euclidean"]
        out["algo"]["random_state"] = [42]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred: {method}")
    return out


def with_search(params: dict, method, **algo_extras) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["search_method"] = [str(method)]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_test(params: dict, choice) -> dict:
    out = copy.deepcopy(params)
    out["test"]["choice"] = [str(choice)]
    return out


def with_ess_window(
    params: dict,
    *,
    ess_fraction: float = 0.25,
    min_window: int = 20,
    max_window: int = 100,
) -> dict:
    out = copy.deepcopy(params)
    out["test"]["window"] = [
        {
            "method": "ess",
            "ess_fraction": ess_fraction,
            "min_window": min_window,
            "max_window": max_window,
        }
    ]
    return out


def with_classical(
    params: dict,
    detector,
    *,
    regimes: int | None = 4,
    min_regime_length: int = 20,
    **algo_extras,
) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.CLASSICAL)]
    out["algo"]["regime_detection_algorithm"] = [str(detector)]
    out["algo"]["dimred"] = ["raw"]
    out["algo"]["feature_map_approx_method"] = ["raw"]
    out["algo"]["post_processing_method"] = ["majority_voting"]
    out["algo"]["min_regime_length"] = [min_regime_length]
    out["algo"]["include_last_regime"] = [True]
    if regimes is not None:
        out["algo"]["regimes"] = [regimes]
    for k, v in algo_extras.items():
        out["algo"][k] = v if isinstance(v, list) else [v]
    return out


def with_model_dimred(params: dict, method, *, regimes: int = 4) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = [str(method)]
    out["algo"]["regimes"] = [regimes]
    return out


def with_tft(
    params: dict,
    *,
    rank: int = 8,
    encoder_length: int = 20,
    prediction_length: int = 5,
    max_epochs: int = 1,
    batch_size: int = 16,
    mode: str = "multivariate",
) -> dict:
    out = copy.deepcopy(params)
    out["algo"]["detection_backend"] = [str(DetectionBackend.KERNEL_RUPTURES)]
    out["algo"]["dimred"] = ["tft"]
    out["algo"]["rank"] = [rank]
    out["algo"]["rank_selection_method"] = ["user_specified"]
    out["algo"]["tft_encoder_length"] = [encoder_length]
    out["algo"]["tft_prediction_length"] = [prediction_length]
    out["algo"]["tft_max_epochs"] = [max_epochs]
    out["algo"]["tft_batch_size"] = [batch_size]
    out["algo"]["tft_mode"] = [mode]
    out["algo"]["num_features"] = [30]
    out["algo"]["depth"] = [1]
    return out


def summarize(res, label: str, df: pl.DataFrame) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def score_vs_truth(res, label: str) -> dict:
    f1 = breakpoint_precision_recall_f1(TRUE_BKPTS, res.bkpts, tolerance=10)
    return {
        "run": label,
        "n_bkpts": len(res.bkpts),
        "bkpts": res.bkpts,
        "covering_vs_truth": covering_metric(TRUE_BKPTS, res.bkpts, features_df.height),
        "ari_vs_truth": adjusted_rand_index(TRUE_BKPTS, res.bkpts, features_df.height),
        "f1_vs_truth": f1["f1"],
        "precision_vs_truth": f1["precision"],
        "recall_vs_truth": f1["recall"],
    }


def run_g1(
    df: pl.DataFrame,
    params: dict,
    label: str,
    plot_vars: list[str],
    *,
    return_fig: bool = False,
):
    """Graph 1 via public single-pass API.

    Returns ``proc`` by default. Pass ``return_fig=True`` to also get the plotnine
    ggplot (without auto-displaying it) for explicit notebook inspection.
    """
    print(
        f"=== Graph 1 · {label} · backend={params['algo'].get('detection_backend')} "
        f"dimred={params['algo']['dimred']} "
        f"detector={params['algo'].get('regime_detection_algorithm')} "
        f"search={params['algo'].get('search_method')} "
        f"test={params['test'].get('choice')} ==="
    )
    proc = run_single_segmentation(df, params)
    summarize(proc, label, df)
    metrics = params.get("metrics", {})
    img_dir = metrics.get("image_dir") or metrics.get("dir")
    if return_fig:
        plot_mode = "write" if img_dir else "display"
        plot_emit = bool(img_dir)
    else:
        plot_mode = "display_and_write" if img_dir else "display"
        plot_emit = True
    fig = plot_regimes(
        df,
        proc,
        variables=plot_vars[:2],
        title=f"Graph 1 · {label}",
        mode=plot_mode,
        img_dir=str(img_dir) if img_dir else None,
        emit=plot_emit,
    )
    if return_fig:
        return proc, fig
    return proc


def run_g2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    plot_vars: list[str],
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
    score_method: str = "mse_to_mean",
    score: dict | None = None,
    return_figs: bool = False,
):
    """Graph 2 via refine_regimes, seeded from a Graph 1 SegmentResults.

    Returns ``out_dir`` by default. Pass ``return_figs=True`` to also get
    ``refined`` and a ``figs`` dict mapping string keys to plotnine ggplots
    (``retrain_iteration_*`` heatmaps + ``regime``).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "score_method": score_method,
        "score": dict(score or {}),
        "regimes_df": None,
    }
    print(f"=== Graph 2 · {label} · score_method={score_method} · seeding from Graph 1 ===")
    print(seed_regimes_from_results(df, seed_res).to_dicts())
    refined = refine_regimes(df, g2, seed=seed_res)
    figs: dict = {}
    if refined is not None:
        summarize(refined, f"{label} Graph 2", df)
        if return_figs:
            figs.update(getattr(refined, "plots", None) or {})
            figs["regime"] = plot_regimes(
                df,
                refined,
                variables=plot_vars[:2],
                title=f"Graph 2 · {label}",
                mode="write",
                img_dir=str(out_dir),
                emit=False,
            )
        else:
            plot_regimes(
                df,
                refined,
                variables=plot_vars[:2],
                title=f"Graph 2 · {label}",
                mode="display",
            )
    if not return_figs:
        pngs = sorted(out_dir.rglob("retrain_iteration_*.png"))[:6]
        print(f"Graph 2 artifacts under {out_dir}")
        for p in pngs:
            print(" ", p.relative_to(out_dir))
            try:
                display(Image(filename=str(p)))
            except Exception as exc:
                print("  (could not display)", exc)
    else:
        print(f"Graph 2 artifacts under {out_dir} ({len(figs)} plotnine figure(s))")
        print(" fig keys:", sorted(figs))
    if return_figs:
        return out_dir, refined, figs
    return out_dir

print("Helpers ready (truth-aware scoring vs TRUE_BKPTS)")


---
# Part A — PCA (baseline)

Default Graph 1: **PCA → RFF → PELT → MMD**, then Graph 2.


## A.1 Graph 1 (PCA)


In [ ]:
params_pca = with_dimred(load_core_params(OUT_DIR / "pca"), "pca")
params_pca["metrics"]["features_to_plot"] = plot_cols
proc_pca, fig_pca = run_g1(features_df, params_pca, "PCA / PELT / MMD", plot_cols, return_fig=True)
fig_pca


## A.2 Graph 2 (seeded from PCA)

Default score `mse_to_mean`. Part N swaps `retrain.score_method`.


In [ ]:
g2_pca_dir, proc_pca_g2, g2_pca_figs = run_g2(
    features_df, params_pca, proc_pca, OUT_DIR / "pca" / "graph2", "PCA", plot_cols, max_iter=3,
    return_figs=True,
)
g2_pca_figs["regime"]


---
# Part B — Kernel PCA


## B.1 Graph 1 (kPCA)


In [ ]:
params_kpca = with_dimred(load_core_params(OUT_DIR / "kpca"), "kpca")
params_kpca["metrics"]["features_to_plot"] = plot_cols
proc_kpca = run_g1(features_df, params_kpca, "kPCA", plot_cols)


## B.2 Graph 2 (seeded from kPCA)


In [ ]:
g2_kpca_dir = run_g2(
    features_df, params_kpca, proc_kpca, OUT_DIR / "kpca" / "graph2", "kPCA", plot_cols, max_iter=2
)


---
# Part C — DMD


## C.1 Graph 1 (DMD)


In [ ]:
params_dmd = with_dimred(load_core_params(OUT_DIR / "dmd"), "dmd")
params_dmd["metrics"]["features_to_plot"] = plot_cols
proc_dmd = run_g1(features_df, params_dmd, "DMD", plot_cols)


## C.2 Graph 2 (seeded from DMD)


In [ ]:
g2_dmd_dir = run_g2(
    features_df, params_dmd, proc_dmd, OUT_DIR / "dmd" / "graph2", "DMD", plot_cols, max_iter=2
)


---
# Part D — t-SNE

Nonlinear manifold embedding (`rank=2`) → RFF → PELT → MMD, then Graph 2.


## D.1 Graph 1 (t-SNE)


In [ ]:
params_tsne = with_dimred(load_core_params(OUT_DIR / "tsne"), "tsne")
proc_tsne = run_g1(features_df, params_tsne, "t-SNE", plot_cols)


## D.2 Graph 2 (seeded from t-SNE)


In [ ]:
g2_tsne_dir = run_g2(
    features_df, params_tsne, proc_tsne, OUT_DIR / "tsne" / "graph2", "t-SNE", plot_cols, max_iter=2
)


---
# Part E — UMAP

Optional `umap-learn` dependency. Skips cleanly when unavailable.


## E.1 Graph 1 (UMAP)


In [ ]:
try:
    import umap  # noqa: F401
    _UMAP_OK = True
except Exception as exc:
    _UMAP_OK = False
    print("UMAP stack unavailable — skipping Part E:", exc)

if _UMAP_OK:
    params_umap = with_dimred(load_core_params(OUT_DIR / "umap"), "umap")
    proc_umap = run_g1(features_df, params_umap, "UMAP", plot_cols)
else:
    proc_umap = proc_pca


## E.2 Graph 2 (seeded from UMAP)


In [ ]:
if _UMAP_OK:
    g2_umap_dir = run_g2(
        features_df, params_umap, proc_umap, OUT_DIR / "umap" / "graph2", "UMAP", plot_cols, max_iter=2
    )
else:
    print("Skipping UMAP Graph 2")


---
# Part F — Search methods (Binseg / BottomUp / WBS / BOCPD)


In [ ]:
params_binseg = with_search(with_dimred(load_core_params(OUT_DIR / "binseg"), "pca"), SearchMethod.BINSEG)
proc_binseg = run_g1(features_df, params_binseg, "Binseg", plot_cols)

params_bottomup = with_search(with_dimred(load_core_params(OUT_DIR / "bottomup"), "pca"), SearchMethod.BOTTOMUP)
proc_bottomup = run_g1(features_df, params_bottomup, "BottomUp", plot_cols)

params_wbs = with_search(
    with_dimred(load_core_params(OUT_DIR / "wbs"), "pca"),
    SearchMethod.WBS,
    wbs_n_intervals=200,
    random_state=42,
)
proc_wbs = run_g1(features_df, params_wbs, "WBS", plot_cols)

params_bocpd = with_search(
    with_dimred(load_core_params(OUT_DIR / "bocpd"), "pca"),
    SearchMethod.BOCPD,
    bocpd_hazard=0.01,
    bocpd_threshold=0.4,
)
proc_bocpd = run_g1(features_df, params_bocpd, "BOCPD", plot_cols)


---
# Part G — Statistical tests


In [ ]:
params_energy = with_test(with_dimred(load_core_params(OUT_DIR / "energy"), "pca"), StatTest.ENERGY_DISTANCE)
proc_energy = run_g1(features_df, params_energy, "energy_distance", plot_cols)

params_mmd_u = with_test(with_dimred(load_core_params(OUT_DIR / "mmd_u"), "pca"), StatTest.MMD_UNBIASED)
proc_mmd_u = run_g1(features_df, params_mmd_u, "mmd_unbiased", plot_cols)

params_hotelling = with_test(with_dimred(load_core_params(OUT_DIR / "hotelling"), "pca"), StatTest.HOTELLING_T2)
proc_hotelling = run_g1(features_df, params_hotelling, "hotelling_t2", plot_cols)

params_mcusum = with_test(
    with_dimred(load_core_params(OUT_DIR / "mcusum"), "pca"), StatTest.MULTIVARIATE_CUSUM
)
proc_mcusum = run_g1(features_df, params_mcusum, "multivariate_cusum", plot_cols)


---
# Part H — ESS window


In [ ]:
params_ess = with_ess_window(with_dimred(load_core_params(OUT_DIR / "ess"), "pca"))
proc_ess = run_g1(features_df, params_ess, "ESS window", plot_cols)


---
# Part I — Classical hard-label detectors

Ask for 4 regimes to match the outer truth (methods still vary in accuracy).


In [ ]:
params_ckmeans = with_classical(
    load_core_params(OUT_DIR / "classical_kmeans"), ClassicalDetector.KMEANS, regimes=4
)
proc_ckmeans = run_g1(features_df, params_ckmeans, "classical kmeans", plot_cols)

params_chmm = with_classical(
    load_core_params(OUT_DIR / "classical_hmm"), ClassicalDetector.HMM, regimes=4
)
proc_chmm = run_g1(features_df, params_chmm, "classical hmm", plot_cols)

params_cjump = with_classical(
    load_core_params(OUT_DIR / "classical_jump"),
    ClassicalDetector.JUMP_MODEL,
    regimes=4,
    jump_penalty=5.0,
)
proc_cjump = run_g1(features_df, params_cjump, "classical jump_model", plot_cols)


## I.4 Graph 2 seeded from classical k-means


In [ ]:
g2_ckmeans_dir = run_g2(
    features_df,
    with_dimred(load_core_params(OUT_DIR / "classical_kmeans" / "graph2"), "pca"),
    proc_ckmeans,
    OUT_DIR / "classical_kmeans" / "graph2",
    "classical kmeans → Graph 2",
    plot_cols,
    max_iter=2,
)


---
# Part J — Classical models as soft dimred


In [ ]:
params_kmeans_dim = with_model_dimred(
    load_core_params(OUT_DIR / "kmeans_dim"), "kmeans", regimes=4
)
proc_kmeans_dim = run_g1(features_df, params_kmeans_dim, "kmeans dimred", plot_cols)

params_hmm_dim = with_model_dimred(load_core_params(OUT_DIR / "hmm_dim"), "hmm", regimes=4)
proc_hmm_dim = run_g1(features_df, params_hmm_dim, "hmm dimred", plot_cols)


---
# Part K — TFT dimred (optional)

Feature columns with `.` are renamed inside `gulfstream.detectors.tft` before
training (no notebook-side sanitization needed).


In [ ]:
try:
    import lightning  # noqa: F401
    import pytorch_forecasting  # noqa: F401

    _TFT_OK = True
except Exception as exc:
    _TFT_OK = False
    print("Skipping TFT:", exc)

if _TFT_OK:
    params_tft = with_tft(load_core_params(OUT_DIR / "tft"), max_epochs=1)
    proc_tft = run_g1(features_df, params_tft, "TFT dimred", plot_cols)
else:
    proc_tft = proc_pca


---
# Part L — ICA / FPCA / factor-style dimred

Not curve-shaped data, but the same dimred knobs still apply.


In [ ]:
params_ica = with_dimred(load_core_params(OUT_DIR / "ica"), "ica")
proc_ica = run_g1(features_df, params_ica, "ICA", plot_cols)

params_fpca = with_dimred(load_core_params(OUT_DIR / "fpca"), "fpca")
proc_fpca = run_g1(features_df, params_fpca, "FPCA", plot_cols)

params_dfactor = with_dimred(load_core_params(OUT_DIR / "dynamic_factor"), "dynamic_factor")
proc_dfactor = run_g1(features_df, params_dfactor, "dynamic_factor", plot_cols)


---
# Part M — Product features (uncertainty, export, events, streaming, panel)


## M.1 Uncertainty bands + CI ribbons


In [ ]:
from gulfstream.metrics import uncertainty as uncertainty_mod

product_dir = OUT_DIR / "product"
product_dir.mkdir(parents=True, exist_ok=True)

params_unc = with_dimred(load_core_params(product_dir / "uncertainty"), "pca")
params_unc["metrics"]["plot_ci_ribbons"] = True
params_unc["uncertainty"] = {
    "enabled": True,
    "sources": ["bootstrap"],
    "level": 0.9,
    "match_tolerance": 5,
    "n_bootstrap": 4,
    "bootstrap_block": 25,
    "random_state": 42,
}
proc_unc = run_g1(features_df, params_unc, "PCA + uncertainty", plot_cols)
proc_unc = uncertainty_mod.evaluate_uncertainty(
    features_df, {**params_unc, "_pipeline_params": params_unc}, proc_unc
)
print("bkpt_ci:", proc_unc.bkpt_ci)
fig_pca_ci_ribbons = plot_regimes(
    features_df, proc_unc, variables=plot_cols[:2], title="PCA + CI ribbons", mode="display"
,
    emit=False,
)
fig_pca_ci_ribbons


## M.2 Excel export + NDJSON events


In [ ]:
import json
import pandas as pd
from gulfstream.metrics.writers import export_breakpoint_excel
from gulfstream.ops.events import emit_run_events

export_dir = product_dir / "export_events"
export_dir.mkdir(parents=True, exist_ok=True)
params_export = copy.deepcopy(params_unc)
params_export["metrics"]["dir"] = str(export_dir)
params_export["metrics"]["image_dir"] = str(export_dir)
params_export["export"] = {
    "excel": {"enabled": True, "dir": str(export_dir), "filename": "bkpt_export.xlsx"}
}
params_export["events"] = {
    "enabled": True, "dir": str(export_dir), "filename": "events.ndjson", "append": False
}
xlsx_path = export_breakpoint_excel(
    proc_unc, params_export, dates=frames.dates_series(features_df).to_list()
)
ndjson_path = emit_run_events(params_export, proc_unc)
print("Excel:", xlsx_path)
print("Events:", ndjson_path)
if ndjson_path:
    for line in Path(ndjson_path).read_text(encoding="utf-8").strip().splitlines():
        print(" ", json.loads(line).get("event"))
if xlsx_path:
    display(pd.read_excel(xlsx_path, sheet_name="Breakpoints"))


## M.3 Streaming Graph 1


In [ ]:
from gulfstream import detect_regimes_incremental

stream_dir = product_dir / "streaming"
stream_dir.mkdir(parents=True, exist_ok=True)
params_stream = with_dimred(load_core_params(stream_dir), "pca")
params_stream["metrics"]["plot"] = False
params_stream["streaming"] = {
    "enabled": True,
    "mode": "expanding",
    "step": 120,
    "min_history": 400,
    "lock_prefix": True,
    "match_tolerance": 5,
}
state = None
proc_stream = None
for step_i in range(3):
    proc_stream, state = detect_regimes_incremental(features_df, params_stream, state)
    print(f"step={step_i} last_t={state.last_t} bkpts={proc_stream.bkpts}")
summarize(proc_stream, "streaming (last step)", features_df)
print(score_vs_truth(proc_stream, "streaming"))


## M.4 Panel joint breakpoints


In [ ]:
from gulfstream import detect_regimes_panel

panel_dir = product_dir / "panel"
panel_dir.mkdir(parents=True, exist_ok=True)
params_panel = with_dimred(load_core_params(panel_dir), "pca")
params_panel["metrics"]["plot"] = False
params_panel["panel"] = {
    "enabled": True,
    "groupby": "columns",
    "combine": "majority",
    "min_group_frac": 0.5,
    "match_tolerance": 5,
}
proc_panel = detect_regimes_panel(features_df, params_panel)
summarize(proc_panel, "panel majority", features_df)
print("panel_support:", proc_panel.panel_support)
print(score_vs_truth(proc_panel, "panel"))


---
# Part N — Graph 2 retrain score methods


## N.1 Compare score matrices on the PCA seed


In [ ]:
import numpy as np
from gulfstream.metrics.regime_scores import known_score_methods, score_feature_regime

score_dir = OUT_DIR / "graph2_scores"
score_dir.mkdir(parents=True, exist_ok=True)
score_df = frames.select_features(features_df, plot_cols)
bkpts_seed = list(proc_pca.bkpts)
feat_names = frames.feature_columns(score_df)
split_kwargs = {"n_splits": 5, "min_side": 10, "max_rows": 80}
specs = [
    ("mse_to_mean", {}),
    ("mad_to_median", {}),
    ("mse_on_diff", {"diff_order": 1}),
    ("factor_residual", {"n_components": 1}),
    ("hotelling_within", {}),
    ("cusum_intensity", {}),
    ("energy_split", split_kwargs),
    ("mmd_split", {**split_kwargs, "mmd_estimator": "linear"}),
]
rows = []
for method, kwargs in specs:
    mat = score_feature_regime(score_df, bkpts_seed, method, **kwargs)
    fi, ri = np.unravel_index(int(np.argmax(mat)), mat.shape)
    rows.append(
        {
            "score_method": method,
            "worst_feature": feat_names[fi],
            "worst_regime": int(ri),
            "max_score": float(mat[fi, ri]),
        }
    )
    print(f"{method:18s} → {feat_names[fi]} @ regime {ri}  (max={mat[fi, ri]:.4g})")
print("Available:", known_score_methods())
score_pick_table = pl.DataFrame(rows)
score_pick_table


## N.2 Graph 2 with alternate scores


In [ ]:
params_g2_scores = with_dimred(load_core_params(score_dir), "pca")
params_g2_scores["metrics"]["plot"] = False

g2_diff_dir = run_g2(
    features_df, params_g2_scores, proc_pca, score_dir / "mse_on_diff",
    "PCA · mse_on_diff", plot_cols, max_iter=2, score_method="mse_on_diff",
    score={"diff_order": 1},
)
g2_energy_dir = run_g2(
    features_df, params_g2_scores, proc_pca, score_dir / "energy_split",
    "PCA · energy_split", plot_cols, max_iter=2, score_method="energy_split",
    score={"n_splits": 5, "min_side": 10, "max_rows": 80},
)
g2_mmd_dir = run_g2(
    features_df, params_g2_scores, proc_pca, score_dir / "mmd_split",
    "PCA · mmd_split", plot_cols, max_iter=2, score_method="mmd_split",
    score={"n_splits": 5, "min_side": 10, "max_rows": 80, "mmd_estimator": "linear"},
)
print("energy:", g2_energy_dir, "mmd:", g2_mmd_dir)


---
# Comparison

Covering, ARI, and breakpoint F1 (tol=10) against **ground-truth** outer breakpoints (and vs PCA as a secondary baseline).


In [ ]:
def row(label: str, res) -> dict:
    out = score_vs_truth(res, label)
    f1_pca = breakpoint_precision_recall_f1(proc_pca.bkpts, res.bkpts, tolerance=10)
    out["covering_vs_pca"] = covering_metric(proc_pca.bkpts, res.bkpts, features_df.height)
    out["ari_vs_pca"] = adjusted_rand_index(proc_pca.bkpts, res.bkpts, features_df.height)
    out["f1_vs_pca"] = f1_pca["f1"]
    return out

rows = [
    {"run": "TRUE", "n_bkpts": len(TRUE_BKPTS), "bkpts": TRUE_BKPTS,
     "covering_vs_truth": 1.0, "ari_vs_truth": 1.0, "f1_vs_truth": 1.0,
     "precision_vs_truth": 1.0, "recall_vs_truth": 1.0,
     "covering_vs_pca": covering_metric(proc_pca.bkpts, TRUE_BKPTS, features_df.height),
     "ari_vs_pca": adjusted_rand_index(proc_pca.bkpts, TRUE_BKPTS, features_df.height),
     "f1_vs_pca": breakpoint_precision_recall_f1(proc_pca.bkpts, TRUE_BKPTS, tolerance=10)["f1"]},
    row("A pca/pelt/mmd", proc_pca),
    row("B kpca", proc_kpca),
    row("C dmd", proc_dmd),
    row("D tsne", proc_tsne),
    row("E umap", proc_umap),
    row("F binseg", proc_binseg),
    row("F bottomup", proc_bottomup),
    row("F wbs", proc_wbs),
    row("F bocpd", proc_bocpd),
    row("G energy", proc_energy),
    row("G mmd_unbiased", proc_mmd_u),
    row("G hotelling_t2", proc_hotelling),
    row("G multivariate_cusum", proc_mcusum),
    row("H ess", proc_ess),
    row("I classical kmeans", proc_ckmeans),
    row("I classical hmm", proc_chmm),
    row("I classical jump_model", proc_cjump),
    row("J kmeans dimred", proc_kmeans_dim),
    row("J hmm dimred", proc_hmm_dim),
    row("K tft", proc_tft),
    row("L ica", proc_ica),
    row("L fpca", proc_fpca),
    row("L dynamic_factor", proc_dfactor),
]
if "proc_unc" in globals():
    rows.append(row("M uncertainty", proc_unc))
if "proc_panel" in globals():
    rows.append(row("M panel", proc_panel))

summary = pl.DataFrame(rows).sort("f1_vs_truth", descending=True)
if "score_pick_table" in globals():
    print("Part N score picks:")
    display(score_pick_table)
print("True bkpts:", TRUE_BKPTS, "sequence:", REGIME_SEQUENCE)
summary


## CLI equivalents

```bash
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/default_core.yaml \
  --source-config config/sources/notebook_faker_hmm.yaml

uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/full_graph2.yaml \
  --source-config config/sources/notebook_faker_hmm.yaml

uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/graph1_export_events.yaml \
  --source-config config/sources/notebook_faker_hmm.yaml
```

Artifacts: `outputs/notebooks/faker_hmm/`.


## What to try next

- Change `seed` / `n_repeating` / `n_features` in the source YAML.
- Raise classical `regimes` mismatch (e.g. ask for 3) to see accuracy drop.
- Graph 2 scores: `config/graph2/graph2_score_{diff,factor,energy,mmd}.yaml`.
- Product YAMLs under `config/graph1/graph1_{streaming,panel,uncertainty,export_events}.yaml`.
